In [1]:
import warnings
warnings.filterwarnings("ignore")

# Getting Dataset

In [7]:
import ccxt
import pandas as pd

exchange = ccxt.poloniex()
symbol = 'BTC/USDT'  # Bitcoin to USDT pair

# Set the desired timeframe 
start_date = pd.Timestamp('2022-12-01')  
end_date = pd.Timestamp('2023-12-01')  

# Convert timestamps to milliseconds (required for Poloniex API)
start_timestamp_ms = int(start_date.timestamp() * 1000)
end_timestamp_ms = int(end_date.timestamp() * 1000)

# Fetch OHLCV (Open, High, Low, Close, Volume) data
ohlcv = exchange.fetch_ohlcv(symbol, timeframe='15m', since=start_timestamp_ms, limit=1000)

# Convert fetched data to a DataFrame
dataframe = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
dataframe['timestamp'] = pd.to_datetime(dataframe['timestamp'], unit='ms')  # Convert timestamp to datetime

dataframe


,timestamp,open,high,low,close,volume
0,2023-12-22 16:00:00,43755.86,43792.60,43670.60,43725.47,40.599877
1,2023-12-22 16:15:00,43725.50,43796.30,43645.99,43697.65,40.999024
2,2023-12-22 16:30:00,43695.68,43713.74,43518.32,43555.71,41.734839
3,2023-12-22 16:45:00,43565.58,43592.79,43503.71,43585.06,38.323177
4,2023-12-22 17:00:00,43545.31,43654.54,43541.08,43625.47,34.881066
...,...,...,...,...,...,...
495,2023-12-27 19:45:00,43079.64,43153.04,43043.78,43148.26,37.288683
496,2023-12-27 20:00:00,43144.51,43197.96,43113.45,43190.25,26.629298
497,2023-12-27 20:15:00,43189.78,43299.94,43127.15,43299.94,39.872256
498,2023-12-27 20:30:00,43299.99,43449.98,43251.48,43449.78,41.108970


In [9]:
from datetime import timedelta
# Calculate the difference in minutes between the start and end timestamps
difference_in_minutes = (end_date - start_date) / timedelta(minutes=1)
# Calculate the number of 15-minute intervals
data_points = difference_in_minutes / 15
print(int(data_points))  # Convert to integer for the exact count of intervals


35040


In [10]:
# Create a date range between start_date and end_date
date_range = pd.date_range(start=start_date, end=end_date, freq='15T').to_frame(index=False, name='timestamp')

# Find missing dates
missing_dates = date_range[~date_range['timestamp'].isin(dataframe['timestamp'])]

print("Dates with missing data:")
print(missing_dates)

Dates with missing data:
                timestamp
0     2022-12-01 00:00:00
1     2022-12-01 00:15:00
2     2022-12-01 00:30:00
3     2022-12-01 00:45:00
4     2022-12-01 01:00:00
...                   ...
35036 2023-11-30 23:00:00
35037 2023-11-30 23:15:00
35038 2023-11-30 23:30:00
35039 2023-11-30 23:45:00
35040 2023-12-01 00:00:00

[35041 rows x 1 columns]


In [11]:
# Identify dates with available data
available_dates = dataframe['timestamp'].dt.date.unique()

# Find the earliest and latest timestamps in the available data
earliest_timestamp = dataframe['timestamp'].min()
latest_timestamp = dataframe['timestamp'].max()

print("Dates with available data:")
print(available_dates)
print("\nTime range with available data:")
print(f"From: {earliest_timestamp}")
print(f"To: {latest_timestamp}")

Dates with available data:
[datetime.date(2023, 12, 22) datetime.date(2023, 12, 23)
 datetime.date(2023, 12, 24) datetime.date(2023, 12, 25)
 datetime.date(2023, 12, 26) datetime.date(2023, 12, 27)]

Time range with available data:
From: 2023-12-22 16:00:00
To: 2023-12-27 20:45:00
